# Tutorial 3: Comparative Analysis for Quality Control

This tutorial demonstrates how to use `boldgenotyper-compare` to detect contamination, mislabeling, and cryptic diversity by comparing species-level and family-level analyses.

## Learning Objectives

By the end of this tutorial, you will:
- Understand the comparative analysis workflow
- Detect contamination and mislabeling in your dataset
- Identify potential cryptic species or hybrids
- Generate sample reassignment recommendations
- Document quality control for publication

## Prerequisites

- Completed Tutorial 1 (Basic Genotyping Workflow)
- Two related datasets:
  - Species-level: `Sphyrna_lewini_scallopedhammerhead.tsv`
  - Family-level: `Sphyrnidae.tsv`

## The Comparative Analysis Concept

By comparing genotype assignments at two taxonomic levels, we can detect:
1. **Contamination**: Species A samples clustering with Species B
2. **Mislabeling**: Samples assigned to wrong species in database
3. **Cryptic diversity**: Distinct genotypes within nominal species
4. **Hybrids**: Samples with intermediate genotypes

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Step 1: Run Species-Level Analysis

First, analyze the species-level dataset (*Sphyrna lewini*).

In [ ]:
# Run species-level analysis
!boldgenotyper ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --clustering-threshold 0.02 \
  --output ../data/Sphyrna_lewini_compare/ \
  --threads 4

## Step 2: Run Family-Level Analysis

Next, analyze the broader family-level dataset (all Sphyrnidae).

In [ ]:
# Run family-level analysis
!boldgenotyper ../data/Sphyrnidae.tsv \
  --clustering-threshold 0.03 \
  --output ../data/Sphyrnidae_compare/ \
  --threads 4

## Step 3: Run Comparative Analysis

Now compare the two analyses to detect quality control issues.

In [ ]:
# Run comparative analysis
!boldgenotyper-compare \
  --species-level ../data/Sphyrna_lewini_compare/ \
  --family-level ../data/Sphyrnidae_compare/ \
  --generate-reassignment-table \
  --output ../data/comparative_analysis/

## Step 4: Examine Output Files

In [ ]:
# List output files
!ls -lh ../data/comparative_analysis/

### Key output files:
1. **comparison_summary.csv** - High-level metrics
2. **genotype_crosswalk.csv** - Species-to-family genotype relationships
3. **sample_reassignments.csv** - Sample-level flagging and recommendations
4. **methods_text.md** - Publication-ready methods section

## Step 5: Review Comparison Summary

In [ ]:
# Load comparison summary
summary = pd.read_csv("../data/comparative_analysis/comparison_summary.csv")

print("="*80)
print("COMPARATIVE ANALYSIS SUMMARY")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

## Step 6: Analyze Genotype Crosswalk

The crosswalk shows how species-level genotypes map to family-level genotypes.

In [ ]:
# Load genotype crosswalk
crosswalk = pd.read_csv("../data/comparative_analysis/genotype_crosswalk.csv")

print("Genotype Crosswalk (first 20 rows):")
print(crosswalk.head(20))

print(f"\nTotal species-level genotypes: {crosswalk['species_genotype'].nunique()}")
print(f"Total family-level genotypes: {crosswalk['family_genotype'].nunique()}")

In [ ]:
# Identify splitting patterns
# How many family genotypes does each species genotype split into?
split_analysis = crosswalk.groupby('species_genotype')['family_genotype'].nunique()

print("\nSplitting pattern analysis:")
print(f"Species genotypes mapping to 1 family genotype: {(split_analysis == 1).sum()}")
print(f"Species genotypes mapping to 2+ family genotypes: {(split_analysis > 1).sum()}")

if (split_analysis > 1).any():
    print("\nSpecies genotypes with multiple family assignments:")
    multi_map = split_analysis[split_analysis > 1]
    print(multi_map.head(10))

In [ ]:
# Visualize genotype relationships
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of samples across genotypes at each level
species_counts = crosswalk.groupby('species_genotype')['sample_count'].sum()
family_counts = crosswalk.groupby('family_genotype')['sample_count'].sum()

axes[0].hist(species_counts.values, bins=30, edgecolor='black', alpha=0.7, label='Species-level')
axes[0].set_xlabel('Samples per Genotype')
axes[0].set_ylabel('Number of Genotypes')
axes[0].set_title('Species-Level Genotype Sizes')
axes[0].set_yscale('log')

axes[1].hist(family_counts.values, bins=30, edgecolor='black', alpha=0.7, 
             color='orange', label='Family-level')
axes[1].set_xlabel('Samples per Genotype')
axes[1].set_ylabel('Number of Genotypes')
axes[1].set_title('Family-Level Genotype Sizes')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## Step 7: Examine Sample Reassignments

This file flags samples that may be contaminated or mislabeled.

In [ ]:
# Load reassignment recommendations
reassignments = pd.read_csv("../data/comparative_analysis/sample_reassignments.csv")

print(f"Total samples evaluated: {len(reassignments)}")
print(f"\nReassignment status:")
print(reassignments['reassignment_flag'].value_counts())

# Show flagged samples
flagged = reassignments[reassignments['reassignment_flag'] == 'FLAGGED']
if len(flagged) > 0:
    print(f"\nFlagged samples: {len(flagged)}")
    print(flagged.head(20))
else:
    print("\nNo samples flagged for reassignment.")

In [ ]:
# Analyze flagged samples by reason
if len(flagged) > 0:
    print("\nFlagging reasons:")
    if 'flag_reason' in flagged.columns:
        print(flagged['flag_reason'].value_counts())
    
    # Geographic distribution of flagged samples
    if 'country' in flagged.columns:
        print("\nFlagged samples by country:")
        print(flagged['country'].value_counts().head(10))

## Step 8: Decision Framework for Flagged Samples

Determine what to do with flagged samples.

In [ ]:
# Create decision summary
if len(flagged) > 0:
    print("="*80)
    print("DECISION FRAMEWORK FOR FLAGGED SAMPLES")
    print("="*80)
    
    for idx, row in flagged.iterrows():
        print(f"\nSample: {row['processid']}")
        print(f"  Original species: {row.get('species_name', 'N/A')}")
        print(f"  Species genotype: {row.get('species_genotype', 'N/A')}")
        print(f"  Family genotype: {row.get('family_genotype', 'N/A')}")
        print(f"  Flag reason: {row.get('flag_reason', 'N/A')}")
        
        # Recommendation
        if 'contamination' in str(row.get('flag_reason', '')).lower():
            print(f"  Recommendation: EXCLUDE from species-level analysis")
        elif 'mislabel' in str(row.get('flag_reason', '')).lower():
            print(f"  Recommendation: VERIFY species ID, may reassign")
        elif 'cryptic' in str(row.get('flag_reason', '')).lower():
            print(f"  Recommendation: INVESTIGATE potential cryptic diversity")
        else:
            print(f"  Recommendation: REVIEW manually")
        
        if idx >= 9:  # Show first 10
            print(f"\n... and {len(flagged) - 10} more flagged samples")
            break
    
    print("\n" + "="*80)
else:
    print("\nNo quality control issues detected. Dataset appears clean.")

## Step 9: Quantify Contamination Rate

In [ ]:
# Calculate contamination statistics
total_samples = len(reassignments)
flagged_samples = len(flagged)
contamination_rate = (flagged_samples / total_samples) * 100 if total_samples > 0 else 0

print("CONTAMINATION ANALYSIS")
print("="*80)
print(f"Total samples analyzed: {total_samples}")
print(f"Flagged samples: {flagged_samples}")
print(f"Contamination rate: {contamination_rate:.2f}%")

if contamination_rate < 1:
    print("\nInterpretation: Very low contamination, excellent data quality")
elif contamination_rate < 5:
    print("\nInterpretation: Low contamination, acceptable for most analyses")
elif contamination_rate < 10:
    print("\nInterpretation: Moderate contamination, review flagged samples")
else:
    print("\nInterpretation: High contamination, careful QC required")

print("="*80)

## Step 10: Generate Publication Methods Text

In [ ]:
# Load methods text
methods_file = "../data/comparative_analysis/methods_text.md"
if Path(methods_file).exists():
    with open(methods_file, 'r') as f:
        methods = f.read()
    print("="*80)
    print("PUBLICATION METHODS TEXT")
    print("="*80)
    print(methods)
    print("="*80)
else:
    print("Methods text file not found.")

## Step 11: Create Quality Control Report

In [ ]:
# Generate comprehensive QC report
qc_report = f"""
QUALITY CONTROL REPORT
Comparative Analysis: Species vs Family Level
{'='*80}

ANALYSIS PARAMETERS
- Species-level dataset: Sphyrna lewini
- Family-level dataset: Sphyrnidae
- Species clustering threshold: 0.02
- Family clustering threshold: 0.03

RESULTS SUMMARY
- Total samples evaluated: {total_samples}
- Samples flagged for review: {flagged_samples}
- Contamination rate: {contamination_rate:.2f}%

GENOTYPE MAPPING
- Species-level genotypes: {crosswalk['species_genotype'].nunique()}
- Family-level genotypes: {crosswalk['family_genotype'].nunique()}
- One-to-one mappings: {(split_analysis == 1).sum()}
- Complex mappings: {(split_analysis > 1).sum()}

QUALITY CONTROL ACTIONS
{'- EXCLUDE from analysis' if flagged_samples > 0 else '- No exclusions needed'}
{'- Review flagged samples manually' if flagged_samples > 0 else ''}
{'- Document exclusions in supplementary materials' if flagged_samples > 0 else ''}

RECOMMENDATIONS
- Dataset quality: {'Excellent' if contamination_rate < 1 else 'Good' if contamination_rate < 5 else 'Acceptable' if contamination_rate < 10 else 'Review required'}
- Proceed with analysis: {'Yes' if contamination_rate < 10 else 'After manual review'}
- Additional validation: {'Optional' if contamination_rate < 5 else 'Recommended'}

{'='*80}
Report generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(qc_report)

# Save report
with open("../data/comparative_analysis/qc_report.txt", 'w') as f:
    f.write(qc_report)
    
print("\nQC report saved to: ../data/comparative_analysis/qc_report.txt")

## Key Takeaways

1. **Multi-Level Validation**: Comparing taxonomic levels detects quality issues
2. **Contamination Detection**: Identifies samples that don't cluster with conspecifics
3. **Objective Criteria**: Automated flagging reduces subjective decisions
4. **Documented QC**: Publication-ready methods and supplementary materials
5. **Improved Reliability**: Excluding flagged samples increases downstream analysis accuracy

## Use Cases

1. **Museum Collections**: Detect mislabeled or mixed specimens
2. **Environmental DNA**: Identify contamination in eDNA samples
3. **Cryptic Diversity**: Discover potential undescribed species
4. **Hybrid Detection**: Identify putative hybrid individuals
5. **Database Curation**: Flag problematic records in BOLD

## Best Practices

1. **Always compare** species vs family/genus when possible
2. **Review flagged samples** manually before excluding
3. **Document decisions** for supplementary materials
4. **Consider biology** - some patterns may be real (hybrids, introgression)
5. **Report contamination rates** in methods section

## Next Steps

- **Tutorial 4**: Use custom shapefiles for non-marine organisms
- **Tutorial 5**: Export for population genetics analysis

## Additional Resources

- Comparative Analysis Guide: `../COMPARATIVE_ANALYSIS_GUIDE.md`
- Use case examples and troubleshooting
- Advanced analysis code snippets